<a href="https://colab.research.google.com/github/SanchsScripts/Plivo_LLM_Submission_Sanchali_23ME31029/blob/main/enzyme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install iterative-stratification
!pip install optuna
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.9 MB/s eta 0:00:00


In [ ]:
import optuna
from optuna.samplers import TPESampler

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedKFold,train_test_split
from sklearn.pipeline import Pipeline

# Models
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold, RepeatedMultilabelStratifiedKFold
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, GradientBoostingClassifier

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
import pandas as pd
import numpy as np
file_names = []
directory = '/kaggle/input'


In [ ]:
train=pd.read_csv("/content/train.csv")
test=pd.read_csv("/content/test.csv")
mixed_desc=pd.read_csv("/content/mixed_desc.csv")

In [ ]:
train.head()

,id,BertzCT,Chi1,Chi1n,Chi1v,Chi2n,Chi2v,Chi3v,Chi4n,EState_VSA1,...,SlogP_VSA3,VSA_EState9,fr_COO,fr_COO2,EC1,EC2,EC3,EC4,EC5,EC6
0,0,323.390782,9.879918,5.875576,5.875576,4.304757,4.304757,2.754513,1.749203,0.000000,...,4.794537,35.527357,0,0,1,1,0,0,0,0
1,1,273.723798,7.259037,4.441467,5.834958,3.285046,4.485235,2.201375,1.289775,45.135471,...,13.825658,44.707310,0,0,0,1,1,0,0,0
2,2,521.643822,10.911303,8.527859,11.050864,6.665291,9.519706,5.824822,1.770579,15.645394,...,17.964475,45.660120,0,0,1,1,0,0,1,0
3,3,567.431166,12.453343,7.089119,12.833709,6.478023,10.978151,7.914542,3.067181,95.639554,...,31.961948,87.509997,0,0,1,1,0,0,0,0
4,4,112.770735,4.414719,2.866236,2.866236,1.875634,1.875634,1.036450,0.727664,17.980451,...,9.589074,33.333333,2,2,1,0,1,1,1,0


In [ ]:
test.head()

,id,BertzCT,Chi1,Chi1n,Chi1v,Chi2n,Chi2v,Chi3v,Chi4n,EState_VSA1,...,PEOE_VSA14,PEOE_VSA6,PEOE_VSA7,PEOE_VSA8,SMR_VSA10,SMR_VSA5,SlogP_VSA3,VSA_EState9,fr_COO,fr_COO2
0,14838,344.632371,7.283603,4.473966,5.834958,3.412257,4.651530,2.096558,1.116433,49.458581,...,13.512441,0.000000,0.000000,0.000000,26.809272,24.539800,4.794537,47.304082,1,1
1,14839,1432.410201,10.663869,7.079026,8.065215,5.297097,5.297097,3.924155,2.569694,0.000000,...,0.000000,34.947374,98.323987,9.606882,0.000000,53.378235,0.000000,43.166667,0,0
2,14840,83.352608,3.931852,1.774215,1.774215,1.073446,1.073446,0.467830,0.170838,5.969305,...,5.969305,0.000000,0.000000,6.420822,11.752550,13.344559,9.589074,24.666667,1,1
3,14841,150.255712,5.912790,3.548812,3.548812,2.595128,2.595128,1.642813,0.694113,0.000000,...,59.935299,0.000000,0.000000,0.000000,17.744066,32.290168,4.794537,26.778866,0,0
4,14842,1817.276351,24.910940,15.540529,20.047314,12.535886,17.730988,11.979618,4.431173,84.554972,...,23.468091,25.609359,0.000000,37.099000,69.141353,38.704130,50.697492,102.583333,0,0


In [ ]:
train.drop(columns=["id"],inplace=True)
test.drop(columns=["id"],inplace=True)
mixed_desc.drop(columns=["CIDs"],inplace=True)
col="EC1_EC2_EC3_EC4_EC5_EC6"

In [ ]:
mixed_desc[col.split("_")]= mixed_desc[col].str.split('_', expand=True).astype(int)
mixed_desc.drop(col, axis=1, inplace=True)

In [ ]:
original = mixed_desc[train.columns]

train = pd.concat([train,original]).reset_index(drop=True)
train.drop(columns=col.split("_")[2:],inplace=True)

In [ ]:
train.head()

,BertzCT,Chi1,Chi1n,Chi1v,Chi2n,Chi2v,Chi3v,Chi4n,EState_VSA1,EState_VSA2,...,PEOE_VSA7,PEOE_VSA8,SMR_VSA10,SMR_VSA5,SlogP_VSA3,VSA_EState9,fr_COO,fr_COO2,EC1,EC2
0,323.390782,9.879918,5.875576,5.875576,4.304757,4.304757,2.754513,1.749203,0.000000,11.938294,...,0.000000,0.000000,17.744066,0.000000,4.794537,35.527357,0,0,1,1
1,273.723798,7.259037,4.441467,5.834958,3.285046,4.485235,2.201375,1.289775,45.135471,0.000000,...,0.000000,0.000000,7.822697,30.705892,13.825658,44.707310,0,0,0,1
2,521.643822,10.911303,8.527859,11.050864,6.665291,9.519706,5.824822,1.770579,15.645394,6.606882,...,53.378235,0.000000,15.645394,73.143616,17.964475,45.660120,0,0,1,1
3,567.431166,12.453343,7.089119,12.833709,6.478023,10.978151,7.914542,3.067181,95.639554,0.000000,...,0.000000,6.420822,15.645394,62.107304,31.961948,87.509997,0,0,1,1
4,112.770735,4.414719,2.866236,2.866236,1.875634,1.875634,1.036450,0.727664,17.980451,12.841643,...,19.386400,0.000000,11.938611,18.883484,9.589074,33.333333,2,2,1,0


In [ ]:
from sklearn.mixture import GaussianMixture
def  get_gmm_class_feature(feat,n):
    gmm=GaussianMixture(n_components=n,random_state=42)
    gmm.fit(train[feat].values.reshape(-1,1))
    train[f'{feat}_class']=gmm.predict(train[feat].values.reshape(-1,1))
    test[f'{feat}_class']=gmm.predict(test[feat].values.reshape(-1,1))

get_gmm_class_feature("BertzCT",4)
get_gmm_class_feature("Chi1",4)
get_gmm_class_feature("Chi1n",3)
get_gmm_class_feature("Chi1v",3)
get_gmm_class_feature("Chi2v",4)
get_gmm_class_feature("Chi3v",3)
get_gmm_class_feature("Chi4n",3)
get_gmm_class_feature("EState_VSA1",2)
get_gmm_class_feature("EState_VSA2",4)
get_gmm_class_feature("ExactMolWt",3)
get_gmm_class_feature("FpDensityMorgan1",3)
get_gmm_class_feature("FpDensityMorgan2",3)
get_gmm_class_feature("FpDensityMorgan3",3)
get_gmm_class_feature("HallKierAlpha",4)
get_gmm_class_feature("HeavyAtomMolWt",3)
get_gmm_class_feature("Kappa3",1)
get_gmm_class_feature("MaxAbsEStateIndex",3)
get_gmm_class_feature("MinEStateIndex",2)
get_gmm_class_feature("NumHeteroatoms",3)
get_gmm_class_feature("PEOE_VSA10",3)
get_gmm_class_feature("PEOE_VSA14",4)
get_gmm_class_feature("PEOE_VSA6",4)
get_gmm_class_feature("PEOE_VSA7",4)
get_gmm_class_feature("PEOE_VSA8",6)
get_gmm_class_feature("SMR_VSA10",2)
get_gmm_class_feature("SMR_VSA5",3)
get_gmm_class_feature("SlogP_VSA3",3)
get_gmm_class_feature("VSA_EState9",3)

In [ ]:
num=['BertzCT', 'Chi1', 'Chi1n', 'Chi1v', 'Chi2n', 'Chi2v', 'Chi3v', 'Chi4n',
       'EState_VSA1', 'EState_VSA2', 'ExactMolWt', 'FpDensityMorgan1',
       'FpDensityMorgan2', 'FpDensityMorgan3', 'HallKierAlpha',
       'HeavyAtomMolWt', 'Kappa3', 'MaxAbsEStateIndex', 'MinEStateIndex',
        'PEOE_VSA10', 'PEOE_VSA14', 'PEOE_VSA6', 'PEOE_VSA7',
       'PEOE_VSA8', 'SMR_VSA10', 'SMR_VSA5', 'SlogP_VSA3', 'VSA_EState9']

train['sum']=train[num].sum(axis=1)
train['mean']=train[num].mean(axis=1)
train['min']=train[num].min(axis=1)
train['max']=train[num].max(axis=1)
train['std']=train[num].std(axis=1)
train['var']=train[num].var(axis=1)

test['sum']=test[num].sum(axis=1)
test['mean']=test[num].mean(axis=1)
test['min']=test[num].min(axis=1)
test['max']=test[num].max(axis=1)
test['std']=test[num].std(axis=1)
test['var']=test[num].var(axis=1)

In [ ]:
def divide_with_check(a,b):
    result = np.where(b != 0, np.divide(a, b), 0)
    return result

def fe(df):
    df['BertzCT_MaxAbsEStateIndex_Ratio']= df['BertzCT'] / (df['MaxAbsEStateIndex'] + 1e-12)
    df['BertzCT_ExactMolWt_Product']= df['BertzCT'] * df['ExactMolWt']
    df['NumHeteroatoms_FpDensityMorgan1_Ratio']= df['NumHeteroatoms'] / (df['FpDensityMorgan1'] + 1e-12)
    df['VSA_EState9_EState_VSA1_Ratio']= df['VSA_EState9'] / (df['EState_VSA1'] + 1e-12)
    df['PEOE_VSA10_SMR_VSA5_Ratio']= df['PEOE_VSA10'] / (df['SMR_VSA5'] + 1e-12)
    df['Chi1v_ExactMolWt_Product']= df['Chi1v'] * df['ExactMolWt']
    df['Chi2v_ExactMolWt_Product']= df['Chi2v'] * df['ExactMolWt']
    df['Chi3v_ExactMolWt_Product']= df['Chi3v'] * df['ExactMolWt']
    df['EState_VSA1_NumHeteroatoms_Product']= df['EState_VSA1'] * df['NumHeteroatoms']
    df['PEOE_VSA10_Chi1_Ratio']= df['PEOE_VSA10'] / (df['Chi1'] + 1e-12)
    df['MaxAbsEStateIndex_NumHeteroatoms_Ratio']= df['MaxAbsEStateIndex'] / (df['NumHeteroatoms'] + 1e-12)
    df['BertzCT_Chi1_Ratio']= df['BertzCT'] / (df['Chi1'] + 1e-12)

In [ ]:
fe(train)
fe(test)

In [ ]:
def generate_features(train, test, cat_cols, num_cols):
    df = pd.concat([train, test], axis = 0, copy = False)
    for c in cat_cols + num_cols:
        df[f'count_{c}'] = df.groupby(c)[c].transform('count')
    for c in cat_cols:
        for n in num_cols:
                df[f'mean_{n}_per_{c}'] = df.groupby(c)[n].transform('median')

    return df.iloc[:len(train),:], df.iloc[len(train):, :]

In [ ]:
target_cols = ['EC1', 'EC2']
cols_to_drop = ['id']

features = [c for c in train.columns if c not in target_cols + cols_to_drop]

cat_cols = ['EState_VSA2','HallKierAlpha','NumHeteroatoms','PEOE_VSA10','PEOE_VSA14','PEOE_VSA6',
            'PEOE_VSA7','PEOE_VSA8', 'SMR_VSA10','SMR_VSA5','SlogP_VSA3','fr_COO','fr_COO2']

num_cols = [c for c in features if c not in cat_cols]

In [ ]:
X_train = train[features]
Y_train = train[target_cols]
X_test = test[features]

In [ ]:
X_train, X_test = generate_features(X_train, X_test, cat_cols, num_cols)

In [ ]:
y  = Y_train
X  = X_train

In [ ]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
#from sklearn.model_selection import RepeatedMultilabelStratifiedKFold
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier


In [ ]:
xgb_params = {'n_estimators': 100,
              'tree_method': 'hist',
              'max_depth': 4,
              'reg_alpha': 0.06790740746476749,
              'reg_lambda': 0.03393770327994609,
              'min_child_weight': 1,
              'gamma': 2.5705812096617772e-05,
              'learning_rate': 0.07132617944894756,
              'colsample_bytree': 0.11664298814833247,
              'colsample_bynode': 0.9912092923877247,
              'colsample_bylevel': 0.29178614622079735,
              'subsample': 0.7395301853144935,
              'random_state': 42
              }


In [ ]:
lgbm_params = {'n_estimators': 200,
 'boosting_type': 'gbdt',
 'max_depth': 10,
 'reg_alpha': 6.720380454685094,
 'reg_lambda': 7.074828689930955e-05,
 'min_child_samples': 15,
 'subsample': 0.5182995486972547,
 'learning_rate': 0.027352422199502537,
 'colsample_bytree': 0.2257179878033366,
 'colsample_bynode': 0.7098194984886731,
 'random_state': 84315}

In [ ]:
xgb_classifier = MultiOutputClassifier(XGBClassifier(**xgb_params))
lgbm_classifier = MultiOutputClassifier(LGBMClassifier(**lgbm_params))

In [ ]:
xgb_clf = Pipeline([('classifier', xgb_classifier)])
lgbm_clf = Pipeline([('classifier', lgbm_classifier)])

In [ ]:
oof_preds_xgb = np.zeros(y.shape)
oof_preds_lgbm = np.zeros(y.shape)

In [ ]:
test_preds_xgb = np.zeros((test.shape[0], y.shape[1]))
test_preds_lgbm = np.zeros((test.shape[0], y.shape[1]))

In [ ]:
oof_losses_xgb = []
oof_losses_lgbm = []

In [ ]:
n_splits = 10
kf = RepeatedMultilabelStratifiedKFold(n_splits=n_splits, n_repeats=1, random_state=42)
train_losses_xgb = []
train_losses_lgbm = []
train_losses_GBC = []

In [ ]:
over_train=[]
over_valid=[]

In [ ]:
for fn, (trn_idx, val_idx) in enumerate(kf.split(X, y)):
    print('Starting fold:', fn)
    X_train, X_val = X.iloc[trn_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[trn_idx], y.iloc[val_idx]

    # Train and predict with XGBoost classifier
    xgb_clf.fit(X_train, y_train)
    train_preds_xgb = xgb_clf.predict_proba(X_train)
    train_preds_xgb = np.array(train_preds_xgb)[:, :, 1].T
    #train_loss_xgb = roc_auc_score(np.ravel(y_train), np.ravel(train_preds_xgb))
    #train_losses_xgb.append(train_loss_xgb)

    val_preds_xgb = xgb_clf.predict_proba(X_val)
    val_preds_xgb = np.array(val_preds_xgb)[:, :, 1].T
    oof_preds_xgb[val_idx] = val_preds_xgb
    loss_xgb = roc_auc_score(np.ravel(y_val), np.ravel(val_preds_xgb))
    oof_losses_xgb.append(loss_xgb)
    preds_xgb = xgb_clf.predict_proba(X_test)
    preds_xgb = np.array(preds_xgb)[:, :, 1].T
    test_preds_xgb += preds_xgb / n_splits



    # Train and predict with LightGBM classifier
    lgbm_clf.fit(X_train, y_train)
    train_preds_lgbm = lgbm_clf.predict_proba(X_train)
    train_preds_lgbm = np.array(train_preds_lgbm)[:, :, 1].T
    #train_loss_lgbm = roc_auc_score(np.ravel(y_train), np.ravel(train_preds_lgbm))
    #train_losses_lgbm.append(train_loss_lgbm)

    val_preds_lgbm = lgbm_clf.predict_proba(X_val)
    val_preds_lgbm = np.array(val_preds_lgbm)[:, :, 1].T
    oof_preds_lgbm[val_idx] = val_preds_lgbm

    loss_lgbm = roc_auc_score(np.ravel(y_val), np.ravel(val_preds_lgbm))
    oof_losses_lgbm.append(loss_lgbm)
    preds_lgbm = lgbm_clf.predict_proba(X_test)
    preds_lgbm = np.array(preds_lgbm)[:, :, 1].T
    test_preds_lgbm += preds_lgbm / n_splits
    """
    # Train and predict with GBC classifier
    GBC_clf.fit(X_train, y_train)
    train_preds_GBC = GBC_clf.predict_proba(X_train)
    train_preds_GBC = np.array(train_preds_GBC)[:, :, 1].T
    #train_loss_lgbm = roc_auc_score(np.ravel(y_train), np.ravel(train_preds_lgbm))
    #train_losses_lgbm.append(train_loss_lgbm)

    val_preds_GBC = GBC_clf.predict_proba(X_val)
    val_preds_GBC = np.array(val_preds_GBC)[:, :, 1].T
    oof_preds_GBC[val_idx] = val_preds_GBC

    loss_GBC = roc_auc_score(np.ravel(y_val), np.ravel(val_preds_lgbm))
    oof_losses_GBC.append(loss_GBC)
    preds_GBC = GBC_clf.predict_proba(X_test)
    preds_GBC = np.array(preds_GBC)[:, :, 1].T
    test_preds_GBC += preds_GBC / n_splits
    """

    overall_train_preds = (train_preds_xgb+train_preds_lgbm)/2
    overall_train_loss = roc_auc_score(np.ravel(y_train), np.ravel(overall_train_preds))
    overall_valid_preds = (val_preds_xgb+val_preds_lgbm)/2
    overall_valid_loss = roc_auc_score(np.ravel(y_val), np.ravel(overall_valid_preds))
    over_train.append(overall_train_loss)
    over_valid.append(overall_valid_loss)
    print("overall_train",overall_train_loss)
    print("overall_valid",overall_valid_loss)

print("over_train",np.mean(over_train))
print("over_valid",np.mean(over_valid))

Starting fold: 0
[LightGBM] [Info] Number of positive: 9507, number of negative: 4783
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.044091 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 80297
[LightGBM] [Info] Number of data points in the train set: 14290, number of used features: 935
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.665290 -> initscore=0.686960
[LightGBM] [Info] Start training from score 0.686960
[LightGBM] [Info] Number of positive: 11372, number of negative: 2918
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.041673 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 80297
[LightGBM] [Info] Number of data points in the train set: 14290, number of used features: 935
[L

In [ ]:
sample_submission.iloc[:,1:] = (test_preds_xgb+test_preds_lgbm)/2

In [ ]:
sample_submission=pd.read_csv("/content/sample_submission.csv")

In [ ]:
sample_submission.to_csv("submission.csv",index=False)